# Re-store an RSNA shard as one file on Kaggle (Phase 1, DEC-17)

Kaggle unpacked our shard tars into folders of JPEGs, and a first read from a folder runs at about 25 slices per second, too slow to train on. This notebook packs one shard back into a single file named `.shard`, which should stay in one piece. The images are copied unchanged, without decoding. The logic is in `src/strokecopilot/data/rsna.py`.

The output holds RSNA data, so this notebook and the dataset made from it stay **private** (`docs/DATA_USE.md`). The printed report holds counts and speeds only.

**Setup**
1. Upload this notebook to Kaggle, or use File → Import Notebook.
2. Add Input: attach `rsna-slim-v1-s0`, `-s1`, `-s2` and `-s3`.
3. Set the accelerator to **None** and turn **Internet on**.

**Run it for shard 0 only, as a test**
1. Leave `SHARD = 0` and click **Save Version → Save & Run All**. Allow up to about 2 hours, which is an upper estimate. You can shut your laptop.
2. Send the report from the last cell.
3. From that version's **Output** tab, create a **new private dataset** named `rsna-slim-v1-file-s0`. Use New Dataset, not a new version of an existing dataset.

Do not run the other shards until Claude has checked that the dataset kept the file whole.


## 1. Settings

In [ ]:
SHARD = 0  # shard 0 first, as a test
DATA_ROOT = '/kaggle/input'
WORK_DIR = '/kaggle/working'
READ_THREADS = 16
REPO_REF = 'main'

## 2. Install the package
From the public repository, at `REPO_REF`; no token is needed.

In [ ]:
import json, os, shutil, subprocess, sys, time

url = f'git+https://github.com/parsa-hke/strokecopilot.git@{REPO_REF}'
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', url], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr[-800:])
del url, result

from strokecopilot.data import prep_rsna, rsna
print('Package installed.')

## 3. Pack the shard
Reads the folder with several threads and prints its progress. This also gives the first true multi-thread speed on a first read.

In [ ]:
shards = rsna.find_shards(DATA_ROOT)
folder, source_index = shards[SHARD]
if not os.path.isdir(folder):
    raise ValueError(f'Shard {SHARD} is already a single file. Nothing to do.')
stem = os.path.basename(folder)
shard_path = os.path.join(WORK_DIR, f'{stem}.shard')
index_path = os.path.join(WORK_DIR, f'{stem}_index.csv')
report = {'shard': SHARD, 'pack': rsna.pack_shard(folder, shard_path, threads=READ_THREADS)}
print(json.dumps(report), flush=True)

## 4. Check it
Copies the index next to the file, then checks every image against it and decodes a sample.

In [ ]:
shutil.copyfile(source_index, index_path)
report['verify'] = prep_rsna.verify_shard(shard_path, index_path)
report['output_files'] = sorted(os.listdir(WORK_DIR))
print(json.dumps(report['verify']), flush=True)

## 5. Report

In [ ]:
text = json.dumps(report, indent=1)
if 'ID_' in text:
    raise ValueError('An ID reached the report. Stop and check.')
print(text)